# 🤖 Bandung AI Travel — Notebook 02: LLM Storyteller
## Integration: RS Output → Groq API → Narasi Perjalanan

**Capstone Project · Telkom University · Program Studi Data Science**

Notebook ini mengambil **output dari Notebook 01** dan menambahkan lapisan **LLM storytelling** via Groq API (free tier).

### Yang dilakukan:
1. ✅ Load model RS (CBF + RL + encoders) dari Notebook 01
2. ✅ Full pipeline: user input → CBF filter → RL selection → TSP route → timestamps
3. ✅ LLM story via Groq: narasi santai, spesifik per destinasi, JSON output
4. ✅ Validasi kontrak frontend React (semua field)
5. ✅ Eksperimen prompt (temperature, bilingual ID/EN)
6. ✅ Export sample JSON siap dipakai backend/frontend

### 📦 Input wajib dari Notebook 01:
| File | Dipakai untuk |
|---|---|
| `data/processed/destinations.csv` | Nama, desc, tags, stay_detail → konteks LLM |
| `data/processed/feature_matrix.npy` | User profile vector CBF |
| `models/cbf_model.pkl` | `similarity_matrix` + `df_index` |
| `models/rl_agent.pkl` | `q_table` + hyperparams |
| `models/label_encoders.pkl` | `feature_cols`, `scaler`, `tfidf` |
| `data/last_updated.txt` | Field `data_last_updated` di response |


## 📦 CELL 1 — Setup & Dependency Check

In [ ]:
# ============================================================
# CELL 1 — Setup & Dependency Check
# ============================================================
import importlib.util, subprocess, sys

REQUIRED = {"requests": "requests", "pandas": "pandas",
            "numpy": "numpy", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import os, re, sys, math, json, time, pickle, urllib.parse
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, requests, warnings
warnings.filterwarnings("ignore")

# ── Pastikan CWD adalah root proyek ─────────────────────────
ROOT_CANDIDATES = [
    Path("/kaggle/working"),
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
    Path.cwd(),
]
ROOT_DIR = next(
    (p for p in ROOT_CANDIDATES if (p / "data/processed/destinations.csv").exists()),
    Path.cwd()
)
os.chdir(ROOT_DIR)
print(f"📂 Working directory: {ROOT_DIR}")

# ── Cek semua file output Notebook 01 ────────────────────────
REQUIRED_FILES = {
    "data/processed/destinations.csv":    "Dataset destinasi",
    "data/processed/feature_matrix.npy": "Feature matrix CBF",
    "models/cbf_model.pkl":              "CBF model",
    "models/rl_agent.pkl":               "RL Agent",
    "models/label_encoders.pkl":         "Label encoders + scalers",
    "data/last_updated.txt":             "Timestamp data",
}
all_ok = True
for path, desc in REQUIRED_FILES.items():
    p = Path(path)
    ok = p.exists()
    kb = p.stat().st_size / 1024 if ok else 0
    status = f"✅ {kb:>7.1f} KB" if ok else "❌ MISSING "
    print(f"  {status}  {path:<45}  {desc}")
    if not ok: all_ok = False

if not all_ok:
    raise FileNotFoundError("⛔ File Notebook 01 tidak lengkap! Jalankan dulu 01_recommendation_engine.ipynb")
print("\n✅ Semua file tersedia.")


## 📂 CELL 2 — Load Models & Data

Penjelasan setiap file output Notebook 01 dan cara penggunaannya di notebook ini.

In [ ]:
# ============================================================
# CELL 2 — Load Models & Data dari Notebook 01
# ============================================================
df = pd.read_csv("data/processed/destinations.csv")
print(f"✅ Dataset: {len(df)} destinasi")
print(f"   Kategori: {df['category'].value_counts().to_dict()}")

def _parse_tags(t):
    if isinstance(t, list): return t
    if pd.isna(t): return []
    s = str(t).strip()
    if not s or s == "[]": return []
    try: return json.loads(s.replace("'", '"'))
    except Exception: return [x.strip().strip("'\"") for x in s.strip("[]").split(",") if x.strip()]

df["tags"] = df["tags"].apply(_parse_tags)
feature_matrix = np.load("data/processed/feature_matrix.npy")
print(f"✅ Feature matrix: {feature_matrix.shape}")

with open("models/label_encoders.pkl", "rb") as f: encoders = pickle.load(f)
print(f"✅ Encoders: {list(encoders.keys())}")

with open("models/cbf_model.pkl", "rb") as f: cbf_data = pickle.load(f)
sim_matrix  = cbf_data["similarity_matrix"]
cbf_df_index = cbf_data["df_index"]
print(f"✅ CBF similarity matrix: {sim_matrix.shape}")

with open("models/rl_agent.pkl", "rb") as f: rl_raw = pickle.load(f)
q_table = defaultdict(lambda: defaultdict(float))
for state, actions in rl_raw["q_table"].items():
    for action_id, q_val in actions.items():
        q_table[state][action_id] = q_val
print(f"✅ RL Agent: {len(q_table)} Q-table states, epsilon={rl_raw.get('epsilon', 0.05)}")

with open("data/last_updated.txt") as f: data_last_updated = f.read().strip()
print(f"✅ Data last updated: {data_last_updated}")

print("\n📊 Statistik dataset:")
print(f"   Harga tiket: Rp {df['ticket'].min():,.0f} – Rp {df['ticket'].max():,.0f}")
print(f"   Rating     : {df['rating'].min():.1f} – {df['rating'].max():.1f} (mean {df['rating'].mean():.2f})")
print(f"   Durasi     : {df['duration'].min()}–{df['duration'].max()} menit")
print(f"   Dengan stay_detail: {df['stay_detail'].notna().sum()}/{len(df)}")

# ── PENJELASAN FIELD-FIELD PENTING DARI NOTEBOOK 01 ────────
print("\n📋 Panduan penggunaan output Notebook 01:")
print("""
  cbf_data['similarity_matrix']  → ndarray (316×316), cosine sim antar destinasi
  cbf_data['df_index']           → list of {id, name, category}, urutan sesuai matrix row
  rl_raw['q_table']              → dict {state_tuple → {dest_id → q_value}}
  rl_raw['epsilon']              → float (0.05 saat produksi = greedy)
  encoders['feature_cols']       → list nama kolom numerik yang dipakai CBF
  encoders['scaler']             → MinMaxScaler fit dari Notebook 01
  encoders['tfidf']              → TfidfVectorizer fit dari Notebook 01
  encoders['category_order']     → ['Alam','Kuliner','Budaya','Wisata','Belanja']
  feature_matrix                 → ndarray (316×F), vektor fitur per destinasi
""")


## ⚙️ CELL 3 — Pipeline Classes

Mirror dari Notebook 01 sehingga notebook ini *self-contained*. Termasuk `ContentBasedFilter`, `rl_select()`, dan `RouteOptimizer`.

In [ ]:
# ============================================================
# CELL 3 — Pipeline Classes (mirror dari Notebook 01)
# Self-contained: tidak perlu import dari notebook lain
# ============================================================
from sklearn.metrics.pairwise import cosine_similarity

CATEGORY_ORDER = ["Alam", "Kuliner", "Budaya", "Wisata", "Belanja"]

def haversine_km(lat1, lng1, lat2, lng2) -> float:
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi, dlam = math.radians(lat2-lat1), math.radians(lng2-lng1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2*R*math.asin(math.sqrt(a))

def minutes_to_hhmm(minutes: int) -> str:
    h, m = divmod(max(0, int(minutes)), 60)
    return f"{h%24:02d}:{m:02d}"


class ContentBasedFilter:
    """
    Wrapper CBF menggunakan similarity_matrix dari Notebook 01.
    Tidak re-compute similarity — langsung pakai hasil pre-compute.
    """
    def __init__(self, sim_matrix, df, feature_matrix, encoders):
        self.sim_matrix   = sim_matrix
        self.df           = df.reset_index(drop=True)
        self.feat_matrix  = feature_matrix
        self.encoders     = encoders

    def recommend(self, categories: list, budget=None, max_km=None,
                  home_lat=-6.9215, home_lng=107.6071, top_n=60):
        """
        Kembalikan top-N kandidat destinasi.
        
        Step 1: Filter hard constraints (kategori, budget, jarak)
        Step 2: Buat user profile vector dari rata-rata fitur destinasi valid
        Step 3: Cosine similarity antara user profile dan semua destinasi
        Step 4: Return top-N sorted by similarity
        """
        # Filter kategori
        mask_cat = self.df["category"].isin(categories) if categories \
                   else pd.Series([True]*len(self.df))
        # Filter budget (harga tiket <= 80% budget)
        mask_budget = (self.df["ticket"] <= budget*0.8) if budget is not None \
                      else pd.Series([True]*len(self.df))
        # Filter radius
        if max_km is not None:
            dist = self.df.apply(
                lambda r: haversine_km(home_lat, home_lng, r["lat"], r["lng"]), axis=1)
            mask_km = dist <= max_km
        else:
            mask_km = pd.Series([True]*len(self.df))

        mask = mask_cat & mask_budget & mask_km
        # Fallback: relax jika terlalu sedikit
        if mask.sum() < max(top_n//3, 5):
            mask = mask_cat
        if mask.sum() == 0:
            mask = pd.Series([True]*len(self.df))

        # User profile = rata-rata fitur destinasi yang lolos filter
        valid_idx    = np.where(mask.values)[0]
        user_profile = self.feat_matrix[valid_idx].mean(axis=0)
        scores       = cosine_similarity([user_profile], self.feat_matrix)[0]
        scores_filt  = scores * mask.values.astype(float)

        top_idx = np.argsort(scores_filt)[::-1][:top_n]
        result  = self.df.iloc[top_idx].copy()
        result["cbf_score"] = scores_filt[top_idx]
        return result[result["cbf_score"] > 0].reset_index(drop=True)


def _state_key(n_selected, spent, budget, cur_time, end_min, selected):
    """
    Encode state ke tuple yang kompatibel dengan Q-table Notebook 01.
    Format: (n_selected_bucket, budget_level, time_level, dominant_cat_idx)
    Sama persis dengan encode di training loop.
    """
    if budget is None or budget >= 999_999_998:
        blevel = 4
    else:
        ratio  = max(0.0, 1.0 - spent/budget) if budget > 0 else 0.0
        blevel = (0 if ratio<=0 else 1 if ratio<0.25 else
                  2 if ratio<0.50 else 3 if ratio<0.75 else 4)
    tleft  = max(0, end_min - cur_time)
    tlevel = (0 if tleft<=0 else 1 if tleft<120 else
              2 if tleft<240 else 3 if tleft<360 else 4)
    dom = 0
    if selected:
        top_cat = Counter(s["category"] for s in selected).most_common(1)[0][0]
        dom = CATEGORY_ORDER.index(top_cat) if top_cat in CATEGORY_ORDER else 0
    return (min(8, n_selected), blevel, tlevel, dom)


def rl_select(candidates_df, q_table, selected, budget, spent, cur_time, end_min):
    """
    Pilih destinasi dengan Q-value tertinggi (inference mode, greedy).
    Fallback: pilih berdasarkan cbf_score jika state tidak ada di Q-table.
    """
    state   = _state_key(len(selected), spent, budget, cur_time, end_min, selected)
    q_state = q_table.get(state, {})
    best_idx, best_q = None, -float("inf")
    for i, row in candidates_df.iterrows():
        q = q_state.get(row["id"], 0.0)
        if q > best_q:
            best_q, best_idx = q, i
    if best_idx is None:
        best_idx = candidates_df["cbf_score"].idxmax()
    return best_idx


class RouteOptimizer:
    """Nearest-neighbor TSP + timestamp builder."""

    def nearest_neighbor_route(self, home, destinations):
        if len(destinations) <= 1: return destinations
        remaining = destinations.copy()
        ordered   = []
        lat, lng  = home["lat"], home["lng"]
        while remaining:
            closest = min(remaining, key=lambda d: haversine_km(lat, lng, d["lat"], d["lng"]))
            ordered.append(closest)
            remaining.remove(closest)
            lat, lng = closest["lat"], closest["lng"]
        return ordered

    def build_itinerary(self, home, home_name, ordered_destinations, start_min, end_min):
        """
        Bangun itinerary dengan timestamps per step.
        Estimasi kecepatan: 30 km/h + overhead macet 10%.
        """
        steps     = []
        cur_time  = start_min
        lat, lng  = home["lat"], home["lng"]
        total_cost, total_km = 0, 0.0

        for idx, dest in enumerate(ordered_destinations, 1):
            km        = haversine_km(lat, lng, dest["lat"], dest["lng"])
            travel_min = max(1, int((km/30)*60*1.1))
            arrive_at  = cur_time + travel_min
            stay_min   = int(dest.get("duration", 90))
            depart_at  = arrive_at + stay_min

            steps.append({
                "idx": idx,
                "dest": {
                    "id":         dest["id"],
                    "name":       dest["name"],
                    "category":   dest["category"],
                    "desc":       dest.get("desc", ""),
                    "ticket":     int(dest.get("ticket", 0)),
                    "duration":   stay_min,
                    "lat":        float(dest["lat"]),
                    "lng":        float(dest["lng"]),
                    "rating":     float(dest.get("rating", 0)),
                    "tags":       dest.get("tags", []),
                    "gmaps_url":  dest.get("gmaps_url", ""),
                    "stay_detail": dest.get("stay_detail", ""),
                },
                "travelMin": travel_min,
                "travelKm":  round(km, 2),
                "arriveAt":  arrive_at,
                "departAt":  depart_at,
            })
            total_cost += int(dest.get("ticket", 0))
            total_km   += km
            cur_time    = depart_at
            lat, lng    = dest["lat"], dest["lng"]

        if ordered_destinations:
            last    = ordered_destinations[-1]
            ret_km  = haversine_km(last["lat"], last["lng"], home["lat"], home["lng"])
            ret_min = max(1, int((ret_km/30)*60*1.1))
        else:
            ret_km, ret_min = 0.0, 0

        return {
            "steps": steps, "totalCost": total_cost,
            "totalKm": round(total_km+ret_km, 2),
            "totalTime": (cur_time+ret_min) - start_min,
            "returnKm": round(ret_km, 2), "returnMin": ret_min,
            "arriveHome": cur_time+ret_min,
            "overBudget": False,
            "spareMin": max(0, end_min-(cur_time+ret_min)),
        }


cbf_model = ContentBasedFilter(sim_matrix, df, feature_matrix, encoders)
optimizer  = RouteOptimizer()
print("✅ ContentBasedFilter, RL inference functions, RouteOptimizer siap.")


## 🔑 CELL 4 — Groq API Setup

### Cara mendapatkan API Key (gratis, tanpa kartu kredit):
1. Buka https://console.groq.com
2. Daftar / login dengan Google
3. **API Keys** → **Create API Key** → copy
4. Simpan di `.env` atau Kaggle Secrets (jangan hardcode di notebook!)

### Batas Free Tier:
| Parameter | Nilai |
|---|---|
| Requests/hari | 14.400 |
| Requests/menit | 30 |
| Token/menit | 131.072 |
| Model default | `llama-3.1-8b-instant` |
| Alternatif | `llama3-70b-8192` (lebih pintar, lebih lambat) |

### Alternatif jika tidak punya Groq:
- **Google Gemini Flash** — https://aistudio.google.com (1500 req/hari gratis)
- **Ollama lokal** — `ollama pull llama3.2` (unlimited, butuh RAM 8GB+)

> ⚠️ **KEAMANAN:** JANGAN commit API key ke Git. Gunakan environment variable atau Kaggle Secrets.


## ⚙️ CELL 5 — Groq Configuration

In [ ]:
# ============================================================
# CELL 5 — Groq API Configuration
# ============================================================
GROQ_API_KEY = None

# 1. Environment variable
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

# 2. File .env
if not GROQ_API_KEY:
    env_path = Path(".env")
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.strip().startswith("GROQ_API_KEY="):
                GROQ_API_KEY = line.split("=",1)[1].strip().strip('"\'')
                break

# 3. Kaggle Secrets
if not GROQ_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
    except Exception:
        pass

# 4. Manual fallback — JANGAN commit ke Git
if not GROQ_API_KEY:
    GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"

GROQ_AVAILABLE = bool(GROQ_API_KEY and GROQ_API_KEY != "YOUR_GROQ_API_KEY_HERE")

if GROQ_AVAILABLE:
    masked = f"{GROQ_API_KEY[:8]}…{GROQ_API_KEY[-4:]}"
    print(f"✅ GROQ_API_KEY: {masked}")
else:
    print("⚠️  GROQ_API_KEY belum diset — akan pakai fallback template.")

GROQ_MODEL       = "llama-3.1-8b-instant"
GROQ_MAX_TOKENS  = 1024
GROQ_TEMPERATURE = 0.75
GROQ_ENDPOINT    = "https://api.groq.com/openai/v1/chat/completions"
print(f"   Model: {GROQ_MODEL} | Tokens: {GROQ_MAX_TOKENS} | Temp: {GROQ_TEMPERATURE}")


## 🗣️ CELL 6 — GroqStoryteller Class

**Perbaikan dari versi baseline:**
- ✅ Prompt lebih kaya: include `stay_detail`, `tags`, `desc`, travel time, arrival time
- ✅ System prompt dipisah untuk konsistensi tone
- ✅ JSON parsing robust (handle markdown code fences dari LLM)
- ✅ Retry logic dengan exponential backoff (1s, 3s, 8s)
- ✅ Fallback template tidak crash pipeline
- ✅ Auto-pad highlights jika count < jumlah step
- ✅ Bilingual (ID/EN) dalam satu class


In [ ]:
# ============================================================
# CELL 6 — GroqStoryteller: Prompt Engineering + API Client
# ============================================================

class GroqStoryteller:
    """
    LLM client untuk menghasilkan narasi perjalanan terstruktur.
    
    Output JSON per request:
      intro      : 1 paragraf santai, maks 80 kata, sebut tiap destinasi
      highlights : list 1 kalimat per destinasi (vivid, spesifik)
      tips       : 2-4 tips praktis yang relevan dengan trip ini
      closing    : 1 kalimat penutup memorable
      vibe       : 2-4 kata feel perjalanan
    
    Perbaikan dari versi baseline:
    + Prompt lebih kaya konteks (stay_detail, tags, desc pendek)
    + System prompt dipisah untuk konsistensi tone
    + JSON parsing robust (handle markdown fences)
    + Retry logic dengan exponential backoff
    + Fallback template tidak crash pipeline
    + Validasi highlights count vs steps count
    + Support bilingual (id/en) dalam satu class
    """

    RETRY_DELAYS = [1, 3, 8]

    def __init__(self, api_key, model="llama-3.1-8b-instant",
                 max_tokens=1024, temperature=0.75):
        self.api_key     = api_key
        self.model       = model
        self.max_tokens  = max_tokens
        self.temperature = temperature
        self.headers     = {"Authorization": f"Bearer {api_key}",
                            "Content-Type": "application/json"}

    def _call_api(self, messages, system_prompt=""):
        payload = {
            "model": self.model,
            "messages": ([{"role":"system","content":system_prompt}] if system_prompt else []) + messages,
            "max_tokens": self.max_tokens,
            "temperature": self.temperature,
        }
        last_err = None
        for i, delay in enumerate(self.RETRY_DELAYS + [None]):
            try:
                resp = requests.post(GROQ_ENDPOINT, headers=self.headers, json=payload, timeout=30)
                if resp.status_code == 429:
                    wait = int(resp.headers.get("Retry-After", delay or 10))
                    print(f"  ⏳ Rate limit. Tunggu {wait}s (attempt {i+1})...")
                    time.sleep(wait); continue
                resp.raise_for_status()
                return resp.json()["choices"][0]["message"]["content"]
            except Exception as e:
                last_err = e
                if delay is not None:
                    print(f"  ⚠️  API error attempt {i+1}: {e}. Retry {delay}s...")
                    time.sleep(delay)
        raise RuntimeError(f"Groq gagal setelah {len(self.RETRY_DELAYS)+1} percobaan: {last_err}")

    @staticmethod
    def _parse_json(raw):
        text = raw.strip()
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
        try: return json.loads(text)
        except json.JSONDecodeError: pass
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try: return json.loads(match.group())
            except Exception: pass
        return None

    @staticmethod
    def _system_prompt(lang):
        if lang == "en":
            return ("You are an upbeat, witty travel blogger writing about Bandung, Indonesia. "
                    "Tone: warm, personal, exciting — like a friend recommending a trip. "
                    "Always respond with a single valid JSON object, no markdown, no preamble.")
        return ("Kamu adalah travel blogger Bandung yang asik, santai, dan penuh semangat. "
                "Gaya bahasa: seperti teman cerita pengalaman liburan — warm, personal, bikin penasaran. "
                "Selalu balas dengan satu JSON object valid, tanpa markdown, tanpa kata pembuka.")

    def _build_prompt(self, itinerary, user_params, lang):
        steps     = itinerary.get("steps", [])
        n         = len(steps)
        start_str = minutes_to_hhmm(user_params.get("startMin", 480))
        end_str   = minutes_to_hhmm(user_params.get("endMin", 1200))
        home_name = user_params.get("homeName", "titik awal")
        budget    = user_params.get("budget")
        budget_str = f"Rp {budget:,.0f}" if budget else "bebas"
        cats      = user_params.get("categories", [])
        vibe_hint = " & ".join(cats) if cats else "Serbaguna"
        total_cost = itinerary.get("totalCost", 0)
        spare_min  = itinerary.get("spareMin", 0)
        over_budget = itinerary.get("overBudget", False)

        # Daftar destinasi lengkap (dipakai LLM untuk menghasilkan cerita spesifik)
        dest_lines = []
        for s in steps:
            d = s["dest"]
            arr = minutes_to_hhmm(s["arriveAt"])
            dep = minutes_to_hhmm(s["departAt"])
            price = f"Rp {d['ticket']:,}" if d["ticket"] > 0 else "Gratis"
            tags  = ", ".join(d["tags"]) if d.get("tags") else "-"
            stay  = d.get("stay_detail") or f"~{d['duration']} menit"
            desc  = (d.get("desc") or "")[:80]
            dest_lines.append(
                f"  {s['idx']}. {d['name']} ({d['category']}) | "
                f"Tiba {arr}, pergi {dep} | perjalanan sebelumnya: {s['travelMin']} menit | "
                f"tiket: {price} | rating: {d['rating']:.1f}/5 | tags: {tags} | "
                f"aktivitas: {stay} | {desc}"
            )
        dest_block = "\n".join(dest_lines)

        json_schema_id = ('{\n'
            '  "intro":      "<satu paragraf santai maks 80 kata, sebut tiap destinasi natural>",\n'
            '  "highlights": ["<1 kalimat vivid per destinasi berurutan>", "..."],\n'
            '  "tips":       ["<tips praktis 1>", "<tips praktis 2>", "<opsional tips 3>"],\n'
            '  "closing":    "<satu kalimat penutup memorable>",\n'
            '  "vibe":       "<2-4 kata feel perjalanan>"\n'
            '}')
        json_schema_en = ('{\n'
            '  "intro":      "<one casual paragraph max 80 words, mention each destination>",\n'
            '  "highlights": ["<1 vivid sentence per destination in order>", "..."],\n'
            '  "tips":       ["<practical tip 1>", "<practical tip 2>", "<optional tip 3>"],\n'
            '  "closing":    "<one memorable closing sentence>",\n'
            '  "vibe":       "<2-4 words capturing the trip feel>"\n'
            '}')

        if lang == "en":
            return (f"Write a Bandung travel story in casual English:\n\n"
                    f"Trip: {home_name} | {start_str}–{end_str} | Budget: {budget_str} "
                    f"| Total ticket: Rp {total_cost:,} | {n} stops | {spare_min} min spare "
                    f"| over budget: {over_budget} | Vibe: {vibe_hint}\n\n"
                    f"Destinations (in order):\n{dest_block}\n\n"
                    f"Respond ONLY with this JSON:\n{json_schema_en}")
        else:
            return (f"Tuliskan cerita wisata Bandung dalam Bahasa Indonesia santai:\n\n"
                    f"Trip: {home_name} | {start_str}–{end_str} | Budget: {budget_str} "
                    f"| Total tiket: Rp {total_cost:,} | {n} destinasi | sisa {spare_min} menit "
                    f"| over budget: {over_budget} | Vibe: {vibe_hint}\n\n"
                    f"Destinasi (urutan kunjungan):\n{dest_block}\n\n"
                    f"Balas HANYA dengan JSON ini:\n{json_schema_id}")

    @staticmethod
    def _fallback(itinerary, user_params, lang):
        steps    = itinerary.get("steps", [])
        names    = [s["dest"]["name"] for s in steps]
        cats     = user_params.get("categories", [])
        vibe     = " & ".join(cats) if cats else "Serbaguna"
        home     = user_params.get("homeName", "titik awal")
        names_str = ", ".join(names[:-1]) + (f", dan {names[-1]}" if len(names)>1 else names[0] if names else "")
        names_str_en = ", ".join(names[:-1]) + (f", and {names[-1]}" if len(names)>1 else names[0] if names else "")
        if lang == "en":
            return {
                "intro":      f"Your Bandung adventure from {home} is all planned! You'll explore {names_str_en}. Get ready for an amazing day!",
                "highlights": [f"Experience the charm of {d} — a must-visit in Bandung." for d in names],
                "tips":       ["Start early to beat the crowd.", "Bring cash for entrance fees.", "Stay hydrated throughout the day."],
                "closing":    "Bandung is calling — let's make unforgettable memories!",
                "vibe":       vibe,
            }
        return {
            "intro":      f"Petualangan Bandung dari {home} sudah tersusun! Kamu akan menjelajahi {names_str}. Bersiaplah untuk hari yang seru!",
            "highlights": [f"Rasakan keunikan {d} — destinasi yang sayang untuk dilewatkan." for d in names],
            "tips":       ["Berangkat pagi biar bebas macet.", "Siapkan uang tunai untuk tiket masuk.", "Jangan lupa sunscreen dan air minum!"],
            "closing":    "Bandung nunggu kamu — yuk bikin kenangan yang nggak terlupakan!",
            "vibe":       vibe,
        }

    def generate_story(self, itinerary, user_params, lang="id"):
        if not self.api_key or self.api_key == "YOUR_GROQ_API_KEY_HERE":
            print("  ⚠️  API key tidak tersedia → fallback template.")
            return self._fallback(itinerary, user_params, lang)
        prompt = self._build_prompt(itinerary, user_params, lang)
        try:
            raw    = self._call_api([{"role":"user","content":prompt}], self._system_prompt(lang))
            parsed = self._parse_json(raw)
            if parsed and all(k in parsed for k in ["intro","highlights","tips","closing","vibe"]):
                # Pastikan highlights count = steps count
                n_steps = len(itinerary.get("steps",[]))
                while len(parsed.get("highlights",[])) < n_steps:
                    parsed["highlights"].append("Destinasi yang tak terlupakan di Bandung.")
                return parsed
            print(f"  ⚠️  JSON tidak lengkap → fallback. Raw[:200]: {raw[:200]}")
            return self._fallback(itinerary, user_params, lang)
        except Exception as e:
            print(f"  ❌ Groq error: {e} → fallback template.")
            return self._fallback(itinerary, user_params, lang)


storyteller = GroqStoryteller(GROQ_API_KEY, GROQ_MODEL, GROQ_MAX_TOKENS, GROQ_TEMPERATURE)
print("✅ GroqStoryteller siap.")


## 🔗 CELL 7 — Full Pipeline: generate_full_itinerary()

Fungsi utama yang menggabungkan semua komponen:
1. **CBF** → filter kandidat by kategori, budget, radius
2. **RL** → pilih destinasi iteratif berdasarkan Q-table
3. **TSP** → urutkan dengan nearest-neighbor
4. **Timestamps** → hitung arriveAt/departAt per step
5. **LLM** → generate story JSON via Groq


In [ ]:
# ============================================================
# CELL 7 — Full Pipeline: generate_full_itinerary()
# ============================================================

def generate_full_itinerary(params: dict, lang: str = "id") -> dict:
    """
    Pipeline lengkap: user params → CBF → RL → Route → LLM → JSON.
    
    Input params (sesuai sample_api_request.json dari Notebook 01):
      home      : {lat, lng}       — titik awal perjalanan
      homeName  : str              — nama titik awal (untuk display & LLM)
      count     : int              — jumlah destinasi yang diinginkan
      maxKm     : float | None     — radius maksimum dari home (km)
      startMin  : int              — waktu mulai (menit dari tengah malam, e.g. 540=09:00)
      endMin    : int              — waktu selesai
      budget    : int | None       — total budget Rupiah (None = unlimited)
      categories: list[str]        — kategori wisata yang diinginkan
    
    Output (sesuai sample_api_response.json):
      steps          : list itinerary dengan timestamps per destinasi
      totalCost      : int (total tiket masuk)
      totalKm        : float (total km termasuk pulang)
      totalTime      : int (total menit dari start sampai tiba home)
      returnKm       : float (km pulang ke home)
      returnMin      : int (estimasi menit pulang)
      arriveHome     : int (menit saat tiba di home)
      overBudget     : bool
      spareMin       : int (sisa waktu setelah tiba home)
      story          : dict {intro, highlights, tips, closing, vibe}
      data_last_updated: str
    """
    home       = params["home"]
    count      = params.get("count", 3)
    budget     = params.get("budget")
    categories = params.get("categories", [])
    start_min  = params.get("startMin", 540)
    end_min    = params.get("endMin", 1200)
    max_km     = params.get("maxKm")
    home_name  = params.get("homeName", "Titik Awal")

    print(f"  📍 Home    : {home_name}")
    print(f"  🏷️  Kategori: {categories}")
    print(f"  💰 Budget  : {'Rp {:,}'.format(budget) if budget else 'Unlimited'}")
    print(f"  📅 Waktu   : {minutes_to_hhmm(start_min)} – {minutes_to_hhmm(end_min)}")
    print(f"  🗺️  Destinasi: {count} | Max radius: {max_km or 'Unlimited'} km")

    # Step 1: CBF filtering → kandidat destinasi
    candidates = cbf_model.recommend(
        categories=categories, budget=budget, max_km=max_km,
        home_lat=home["lat"], home_lng=home["lng"], top_n=60
    )
    print(f"  ✅ CBF: {len(candidates)} kandidat")

    # Step 2: RL selection → pilih destinasi secara iteratif
    selected = []
    spent    = 0
    cur_time = start_min
    pool     = candidates.copy()

    for _ in range(count):
        if pool.empty: break
        idx  = rl_select(pool, q_table, selected, budget, spent, cur_time, end_min)
        dest = pool.loc[idx].to_dict()
        selected.append(dest)
        spent    += int(dest.get("ticket", 0))
        cur_time += int(dest.get("duration", 90)) + 20
        pool      = pool.drop(index=idx)

    if not selected:
        selected = candidates.head(count).to_dict("records")
        print(f"  ⚠️  RL fallback: pakai top-{count} CBF score")
    else:
        print(f"  ✅ RL: {len(selected)} destinasi terpilih")

    # Step 3: Route optimization (nearest-neighbor TSP)
    ordered = optimizer.nearest_neighbor_route(home, selected)
    print(f"  ✅ Route dioptimasi (TSP nearest-neighbor)")

    # Step 4: Build itinerary dengan timestamps
    itinerary = optimizer.build_itinerary(
        home=home, home_name=home_name,
        ordered_destinations=ordered,
        start_min=start_min, end_min=end_min
    )
    itinerary["overBudget"] = bool(budget and itinerary["totalCost"] > budget)

    print(f"  ✅ Itinerary: Rp {itinerary['totalCost']:,} | "
          f"{itinerary['totalKm']:.1f} km | "
          f"{'⚠️ OVER BUDGET' if itinerary['overBudget'] else '✅ In budget'} | "
          f"Sisa {itinerary['spareMin']} menit")

    # Step 5: LLM storytelling
    print(f"  🤖 LLM story ({lang})...")
    story = storyteller.generate_story(itinerary, params, lang)
    itinerary["story"]              = story
    itinerary["data_last_updated"]  = data_last_updated

    return itinerary


print("✅ generate_full_itinerary() siap.")


## 🔌 CELL 8 — Test Koneksi Groq API

In [ ]:
# ============================================================
# CELL 8 — Test Koneksi Groq API
# ============================================================
print("🔌 Testing Groq API...")
if not GROQ_AVAILABLE:
    print("⚠️  API Key belum diset — skip (fallback template aktif).")
else:
    try:
        raw = storyteller._call_api(
            [{"role":"user","content":'Respond ONLY with valid JSON: {"status":"ok","model":"test"}'}],
            "You are a test API. Respond only with valid JSON."
        )
        result = storyteller._parse_json(raw)
        if result and result.get("status") == "ok":
            print(f"✅ Groq API OK! Model: {GROQ_MODEL}")
        else:
            print(f"⚠️  Response tidak expected tapi API merespons: {raw[:100]}")
    except Exception as e:
        print(f"❌ Groq API GAGAL: {e}")
        print("   Pipeline tetap jalan dengan fallback template.")


## 🧪 CELL 9 — Full Integration Test (3 Skenario)

Tiga skenario berbeda: alam saja, kuliner+budaya dengan radius terbatas, dan multi-kategori tanpa batas budget (output English).

In [ ]:
# ============================================================
# CELL 9 — Full Integration Test (3 Skenario)
# ============================================================
test_cases = [
    {
        "name": "Wisata Alam Selatan Bandung (Budget Ketat)",
        "params": {
            "home": {"lat":-6.9215,"lng":107.6071}, "homeName":"Alun-Alun Bandung",
            "count":3, "maxKm":None, "startMin":7*60, "endMin":18*60,
            "budget":300_000, "categories":["Alam"],
        },
        "lang": "id",
    },
    {
        "name": "City Tour Kuliner & Budaya (Radius 20km)",
        "params": {
            "home": {"lat":-6.9145,"lng":107.6020}, "homeName":"Stasiun Bandung",
            "count":4, "maxKm":20, "startMin":10*60, "endMin":21*60,
            "budget":500_000, "categories":["Kuliner","Budaya"],
        },
        "lang": "id",
    },
    {
        "name": "Weekend Adventure — No Budget Limit (English)",
        "params": {
            "home": {"lat":-6.8126,"lng":107.6178}, "homeName":"Pasar Lembang",
            "count":5, "maxKm":None, "startMin":8*60, "endMin":20*60,
            "budget":None, "categories":["Alam","Wisata","Kuliner"],
        },
        "lang": "en",
    },
]

results = []
for tc in test_cases:
    print(f"\n{'='*65}\n🧪 {tc['name']}\n{'='*65}")
    res = generate_full_itinerary(tc["params"], lang=tc["lang"])
    results.append(res)

    print(f"\n📍 Itinerary ({len(res['steps'])} destinasi):")
    for s in res["steps"]:
        arr = minutes_to_hhmm(s["arriveAt"])
        dep = minutes_to_hhmm(s["departAt"])
        print(f"   {s['idx']}. {s['dest']['name']:<30} | {arr}–{dep} | "
              f"{s['travelKm']:.1f}km | Rp{s['dest']['ticket']:>7,} | ⭐{s['dest']['rating']:.1f}")

    budget = tc["params"].get("budget")
    print(f"\n💰 Total tiket: Rp {res['totalCost']:,} / "
          f"{'Rp {:,}'.format(budget) if budget else 'Unlimited'} "
          f"{'⚠️ OVER' if res['overBudget'] else '✅ OK'}")
    print(f"🗺️  Total km   : {res['totalKm']:.1f} | Pulang: {res['returnKm']:.1f}km ~{res['returnMin']}mnt")
    print(f"⏰ Tiba home  : {minutes_to_hhmm(res['arriveHome'])} (sisa {res['spareMin']}mnt)")

    story = res.get("story", {})
    print(f"\n✨ Vibe   : {story.get('vibe','-')}")
    print(f"📖 Intro  : {story.get('intro','-')}")
    for i,h in enumerate(story.get("highlights",[]),1):
        print(f"   [{i}] {h}")
    for tip in story.get("tips",[]):
        print(f"   💡 {tip}")
    print(f"🎯 Closing: {story.get('closing','-')}")


## ✅ CELL 10 — Validasi Kontrak Frontend React

Pastikan setiap field yang dibutuhkan frontend tersedia dan tipenya benar.

In [ ]:
# ============================================================
# CELL 10 — Validasi Kontrak Frontend React
# ============================================================
def validate_frontend_contract(response, test_name=""):
    errors = []
    ROOT_FIELDS = {
        "steps":(list,), "totalCost":(int,float), "totalKm":(int,float),
        "totalTime":(int,float), "returnKm":(int,float), "returnMin":(int,float),
        "arriveHome":(int,float), "overBudget":(bool,), "spareMin":(int,float),
        "story":(dict,), "data_last_updated":(str,),
    }
    for field, types in ROOT_FIELDS.items():
        if field not in response:
            errors.append(f"Missing root: {field}")
        elif not isinstance(response[field], types):
            errors.append(f"Wrong type {field}: expected {types}, got {type(response[field])}")

    DEST_FIELDS   = ["id","name","category","desc","ticket","duration","lat","lng","rating","tags","gmaps_url"]
    VALID_CATS    = {"Alam","Kuliner","Budaya","Wisata","Belanja"}
    STORY_FIELDS  = ["intro","highlights","tips","closing","vibe"]

    for i, step in enumerate(response.get("steps",[])):
        for f in ["idx","dest","travelMin","travelKm","arriveAt","departAt"]:
            if f not in step: errors.append(f"steps[{i}] missing: {f}")
        if "dest" in step:
            for f in DEST_FIELDS:
                if f not in step["dest"]: errors.append(f"steps[{i}].dest missing: {f}")
            cat = step["dest"].get("category","")
            if cat not in VALID_CATS: errors.append(f"steps[{i}].dest.category invalid: '{cat}'")

    story   = response.get("story", {})
    n_steps = len(response.get("steps",[]))
    for f in STORY_FIELDS:
        if f not in story: errors.append(f"story missing: {f}")
    n_hl = len(story.get("highlights",[]))
    if n_hl != n_steps:
        errors.append(f"highlights ({n_hl}) != steps ({n_steps})")

    prefix = f"[{test_name[:40]}] " if test_name else ""
    if errors:
        print(f"  ❌ {prefix}GAGAL ({len(errors)} error):")
        for e in errors: print(f"     - {e}")
        return False
    print(f"  ✅ {prefix}Kontrak OK")
    return True

print("🔍 Validasi semua hasil test:\n")
for res, tc in zip(results, test_cases):
    validate_frontend_contract(res, tc["name"])

print("\n📊 Summary:")
for i,(res,tc) in enumerate(zip(results,test_cases)):
    s = res["story"]
    ok = all(k in s for k in ["intro","highlights","tips","closing","vibe"])
    print(f"  {i+1}. {tc['name'][:45]:<45} | Story: {'✅' if ok else '❌'} | "
          f"Steps: {len(res['steps'])} | Rp {res['totalCost']:,}")


## 🌐 CELL 11 — Bilingual Test (ID vs EN)

In [ ]:
# ============================================================
# CELL 11 — Bilingual Test (ID vs EN)
# ============================================================
print("=== BILINGUAL TEST ===\n")
bilingual_params = {
    "home":{"lat":-6.9215,"lng":107.6071}, "homeName":"Alun-Alun Bandung",
    "count":3, "maxKm":None, "startMin":9*60, "endMin":19*60,
    "budget":400_000, "categories":["Alam","Kuliner"],
}
for lang in ["id","en"]:
    label = "Indonesia 🇮🇩" if lang=="id" else "English 🇬🇧"
    print(f"\n--- {label} ---")
    res   = generate_full_itinerary(bilingual_params, lang=lang)
    story = res["story"]
    print(f"Vibe   : {story['vibe']}")
    print(f"Intro  : {story['intro']}")
    if story["tips"]: print(f"Tip #1 : {story['tips'][0]}")
    print(f"Closing: {story['closing']}")
    time.sleep(2)


## 🎛️ CELL 12 — Eksperimen Temperature

Bandingkan output temperature 0.3, 0.7, 1.0 pada itinerary yang sama untuk menemukan setting optimal produksi.

In [ ]:
# ============================================================
# CELL 12 — Eksperimen Temperature
# ============================================================
print("=== EKSPERIMEN TEMPERATURE ===\n")
if not GROQ_AVAILABLE:
    print("⚠️  API Key belum diset — skip.")
else:
    exp_params = {
        "home":{"lat":-6.9215,"lng":107.6071}, "homeName":"Alun-Alun Bandung",
        "count":2, "maxKm":None, "startMin":9*60, "endMin":17*60,
        "budget":200_000, "categories":["Alam"],
    }
    exp_res = generate_full_itinerary(exp_params, lang="id")
    print("Itinerary sama, hanya temperature berbeda:\n")
    for temp in [0.3, 0.7, 1.0]:
        ts    = GroqStoryteller(GROQ_API_KEY, GROQ_MODEL, GROQ_MAX_TOKENS, temp)
        story = ts.generate_story(exp_res, exp_params, lang="id")
        print(f"Temperature {temp}:")
        print(f"  Vibe : {story.get('vibe','-')}")
        print(f"  Intro: {story.get('intro','-')[:120]}...")
        print()
        time.sleep(2)

    print("📌 Rekomendasi: temperature=0.7 untuk produksi (konsisten + kreatif)")


## 💾 CELL 13 — Export Sample Output untuk Backend Developer

In [ ]:
# ============================================================
# CELL 13 — Export Sample Output untuk Backend & Frontend Dev
# ============================================================
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

sample_out = results[0]
sample_in  = test_cases[0]["params"]

with open(out_dir/"sample_api_response.json","w",encoding="utf-8") as f:
    json.dump(sample_out, f, ensure_ascii=False, indent=2, default=str)
print("✅ data/processed/sample_api_response.json")

with open(out_dir/"sample_api_request.json","w",encoding="utf-8") as f:
    json.dump(sample_in, f, ensure_ascii=False, indent=2)
print("✅ data/processed/sample_api_request.json")

print("\n📄 Preview response (field summary):")
for k,v in sample_out.items():
    if k == "steps":
        print(f"  steps          : list[{len(v)}] itinerary steps")
    elif k == "story":
        print(f"  story          : {list(v.keys())}")
    else:
        print(f"  {k:<18}: {v}")

print("\n📋 Story output:")
s = sample_out.get("story",{})
print(f"  vibe      : {s.get('vibe','')}")
print(f"  intro     : {s.get('intro','')}")
for i,h in enumerate(s.get("highlights",[]),1):
    print(f"  highlight {i}: {h}")
for tip in s.get("tips",[]):
    print(f"  tip       : {tip}")
print(f"  closing   : {s.get('closing','')}")


## ✅ Notebook 02 Selesai

### Ringkasan Status
| Komponen | Status | Keterangan |
|---|---|---|
| Load model RS | ✅ | CBF sim_matrix + Q-table RL + encoders |
| CBF filtering | ✅ | Filter kategori, budget 80%, radius km, fallback |
| RL selection | ✅ | Q-table greedy inference, state encoding identik training |
| TSP route | ✅ | Nearest-neighbor dari home, timestamps per step |
| Groq LLM | ✅ | llama-3.1-8b-instant, JSON structured, retry 3x |
| Fallback | ✅ | Pipeline tidak crash jika API gagal |
| Bilingual | ✅ | Indonesia & English, prompt terpisah |
| Frontend contract | ✅ | Semua field tervalidasi incl. highlights count |
| Export | ✅ | sample_api_request.json + sample_api_response.json |

### Output File Notebook 02
| File | Digunakan oleh |
|---|---|
| `data/processed/sample_api_response.json` | Backend dev & frontend mock |
| `data/processed/sample_api_request.json` | Dokumentasi API schema |

### Langkah Selanjutnya
1. **Backend FastAPI** — bungkus `generate_full_itinerary()` sebagai endpoint `POST /api/recommend`
2. **Frontend React** — form input + komponen `RouteTimeline` (rute putus-putus) + story panel
3. **Deployment** — VPS + domain .my.id + Nginx + Certbot SSL

### Catatan Groq Free Tier
- Monitor usage di https://console.groq.com/usage
- Tambahkan caching (e.g. Redis) untuk request berulang di production
- `llama3-70b-8192` tersedia bila butuh kualitas narasi lebih tinggi
